In [1]:
import pandas as pd
import numpy as np


In [3]:
df=pd.read_csv('../data/interim/listings_parsed_v7.csv', dtype={"jnc": str})    


In [10]:

intervals=pd.read_csv('../data/interim/quantile_intervals_groupkfold.csv', index_col=0)

comps=pd.read_csv('../data/interim/nn_comps.csv', index_col=0)

In [ ]:
def build_valuation_table(df, intervals, comps):
    base_cols = [
        "jnc", "Price_yen", "municipality", "area_m2", "age_years",
        "walk_min_nearest", "n_rooms", "building_key", "lookalike_id"
    ]

    interval_cols = ["lo2", "hi2", "mid"]
    if not set(interval_cols).issubset(intervals.columns):
        interval_cols = ["lo", "hi", "mid"]

    val = (
        df[base_cols]
        .join(intervals[interval_cols])
        .join(comps[["comp_est", "comp_lo", "comp_hi", "comp_n"]])
        .rename(columns={
            interval_cols[0]: "lo",
            interval_cols[1]: "hi",
            "mid": "estimate",
        })
    )

    # Convert log-space predictions to yen if necessary.
    if val[["lo", "estimate", "hi"]].median().median() < 30:
        val[["lo", "estimate", "hi"]] = np.expm1(
            val[["lo", "estimate", "hi"]]
        )

    assert len(val) == len(df)
    assert val[["estimate", "lo", "hi"]].notna().all().all()
    assert (val["lo"] <= val["estimate"]).all()
    assert (val["estimate"] <= val["hi"]).all()

    val["position"] = np.select(
        [
            val["Price_yen"] > val["hi"],
            val["Price_yen"] < val["lo"],
        ],
        ["above", "below"],
        default="within",
    )

    val["gap_pct"] = (val["Price_yen"] - val["estimate"]) / val["estimate"]

    val["comp_position"] = np.select(
        [
            val["Price_yen"] > val["comp_hi"],
            val["Price_yen"] < val["comp_lo"],
        ],
        ["above", "below"],
        default="within",
    )

    val["region"] = np.select(
        [
            val["municipality"].astype(str).str.endswith("区"),
            val["municipality"].astype(str).str.endswith("市"),
        ],
        ["ward", "city"],
        default="town_village",
    )

    val["confidence"] = np.where(
        (val["region"] == "town_village")
        | (val["area_m2"] > 80)
        | (val["comp_n"] < 5),
        "low",
        "normal",
    )

    val["label"] = np.select(
        [
            (val["position"] == "above") & (val["comp_position"] == "above"),
            (val["position"] == "above"),
            (val["position"] == "below") & (val["comp_position"] == "below"),
            (val["position"] == "below"),
        ],
        [
            "Relatively high vs model and similar listings",
            "Above the model's expected range",
            "Relatively low vs model and similar listings",
            "Below the model's expected range",
        ],
        default="Within the expected range",
    )

    return val


val = build_valuation_table(df, intervals, comps)

print("Rows:", len(val))
print("\nNaN counts:")
print(val.isna().sum())

print("\nPosition shares (%):")
print(val["position"].value_counts(normalize=True).mul(100).round(2))

print("\nLow-confidence share (%):")
print((val["confidence"].eq("low").mean() * 100).round(2))

sample_cols = (
    val.select_dtypes(exclude=["object", "string", "category"])
    .columns.difference(["jnc", "building_key"], sort=False)
)

print("\nRandom sample:")
display(val.sample(5, random_state=42)[sample_cols])

val.to_csv("../data/interim/valuation_table.csv", index=True)